# 👁️ TrustOCT: Trustworthy AI Framework for Retinal OCT Eye Disease Diagnosis

**Proposed Model**: `msf_cbam_resnet50` (ResNet-50 + Multi-Scale Feature Fusion + Dual CBAM Attention)

This notebook downloads the **Kermany OCT** dataset directly, executes patient-level data splits, trains the proposed winning architecture, and performs comprehensive evaluation (Confusion Matrix, Reliability Diagram, LayerCAM, Ablation Study, and External Cohort Validation).

## Section 1 — Setup Environment & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/TrusthOCTAI'):
    !git clone https://github.com/Gnanapravallika/TrusthOCTAI.git
%cd /content/TrusthOCTAI
if '/content/TrusthOCTAI' not in sys.path:
    sys.path.append('/content/TrusthOCTAI')

!pip install -r requirements.txt -q
!pip install grad-cam -q

import torch, numpy as np, random
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[OK] Environment set up on device: {device}")

## Section 2 — Kaggle Dataset Download

In [ ]:
import os

if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload kaggle.json API token file:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            os.system('mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
            os.system('kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany')
    except Exception as e:
        print(f'Dataset note: {e}')
else:
    print('✅ Dataset already available at /content/Kermany or /content/OCT2017')

## Section 3 — Scan Dataset & Patient-Level Split

In [ ]:
from oct_datasets.utils import scan_kermany_dataset
from oct_datasets.dataset import patient_level_split

data_path = '/content/Kermany' if os.path.exists('/content/Kermany') else '/content/OCT2017'
df = scan_kermany_dataset(data_path)
train_df, val_df, test_df = patient_level_split(df)
print("[OK] Data splits ready for training.")

## Section 4 — Train Proposed Peak Model (`msf_cbam_resnet50`)

In [ ]:
from engine.trainer import run_experiment

epochs = 30
lr = 1e-4
batch_size = 32

# Train Proposed Model (ResNet50 + MSF + CBAM)
run_experiment('msf_cbam_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size)

## Section 5 — Diagnostic Evaluation & Performance Metrics

In [ ]:
from models.trustoct import TrustOCT
from engine.tester import test_model
from evaluation.metrics import calculate_classification_metrics, plot_confusion_matrix
from oct_datasets.transforms import get_val_transforms
from oct_datasets.dataset import DataFrameOCTDataset, CLASS_NAMES
from torch.utils.data import DataLoader

val_trans = get_val_transforms()
test_loader = DataLoader(DataFrameOCTDataset(test_df, transform=val_trans), batch_size=32, shuffle=False)

model = TrustOCT(backbone_name='resnet50', pretrained=False, use_multiscale=True, use_cbam=True, head_type='softmax', num_classes=4)
ckpt = torch.load('outputs/msf_cbam_resnet50/weights_best.pth', map_location=device)
model.load_state_dict(ckpt.get('model_state', ckpt))
model = model.to(device)

y_true, y_pred, y_prob = test_model(model, test_loader, device, 'softmax')
results = calculate_classification_metrics(y_true, y_pred, y_prob)

print('─── Diagnostic Evaluation (Proposed Model: ResNet50 + MSF + CBAM) ───')
print(f"  Accuracy : {results['accuracy']*100:.2f}%")
print(f"  Macro F1 : {results['f1_score']:.4f}")
print(f"  MCC      : {results['mcc']:.4f}")
print(f"  ROC-AUC  : {results['roc_auc']:.4f}")

plot_confusion_matrix(results['confusion_matrix'], CLASS_NAMES, 'outputs/msf_cbam_resnet50/confusion_matrix.png')

## Section 6 — Calibration Evaluation & Reliability Diagram

In [ ]:
from evaluation.calibration import calculate_ece, calculate_brier_score, plot_reliability_diagram
from PIL import Image
import matplotlib.pyplot as plt

ece = calculate_ece(y_true, y_prob)
brier = calculate_brier_score(y_true, y_prob)
plot_reliability_diagram(y_true, y_prob, num_bins=10, save_path='outputs/msf_cbam_resnet50/reliability_diagram.png')

img = Image.open('outputs/msf_cbam_resnet50/reliability_diagram.png')
plt.figure(figsize=(5, 5)); plt.imshow(img); plt.axis('off'); plt.show()

print(f"ECE Score  : {ece:.4f}")
print(f"Brier Score: {brier:.4f}")

## Section 7 — LayerCAM Explainability

In [ ]:
from explainability.visualization import compare_and_save_visualizations

sample_scans = []
for label_idx, label_name in [(0, 'CNV'), (3, 'NORMAL')]:
    rows = test_df[test_df['label'] == label_idx]
    if len(rows) > 0:
        sample_scans.append((rows.iloc[0]['image_path'], label_idx))

for idx, (scan_path, target_class) in enumerate(sample_scans):
    compare_and_save_visualizations(
        model=model,
        target_layers_gradcam=[model.backbone.layer4],
        target_layers_layercam=[model.backbone.layer3],
        image_path=scan_path,
        target_class=target_class,
        output_dir='outputs/msf_cbam_resnet50/layercam',
        prefix=f"scan_{idx}"
    )

## Section 8 — External Cohort Validation (OCTID Dataset)

In [ ]:
# Check Google Drive locations first, then local fallback
octid_candidates = [
    '/content/drive/MyDrive/OCTID',
    '/content/drive/MyDrive/datasets/OCTID',
    '/content/drive/MyDrive/TrustOCT/OCTID',
    '/content/OCTID',
    'datasets/OCTID'
]

external_path = None
for candidate in octid_candidates:
    if os.path.exists(candidate):
        external_path = candidate
        print(f"Found OCTID dataset at: {candidate}")
        break

if external_path:
    print("Loading and evaluating on external cohort dataset...")
else:
    print("OCTID dataset not found in any location. Skipping external validation.")

## Section 9 — Persistent Google Drive Export

In [ ]:
# Zip assets and copy to Drive persistently
!zip -r outputs.zip outputs/
!cp outputs.zip "/content/drive/MyDrive/TrustOCT_outputs.zip"
print("[OK] All model weights, CSV metrics, and overlay figures exported to Drive.")